# PyTorch extension: predicting TMPRSS2–ERG status

I added this small neural-network experiment **after the master's thesis** while learning PyTorch. It was not part of the submitted thesis.

## Study design

- **Development cohort:** TCGA PanCancer Atlas
- **Early-stopping validation:** a stratified 20% TCGA subset used only for model selection
- **Internal test:** a separate stratified 20% TCGA subset, untouched until evaluation
- **External evaluation:** SU2C/PCF Dream Team 2019 capture cohort, never used during fitting or model selection
- **Primary features:** RAI2, CTBP1 and CTBP2
- **Positive control:** ERG plus the three primary features

The main model excludes ERG because its expression is a direct marker of TMPRSS2–ERG and would make the task close to trivial. I kept a second, ERG-inclusive model as a control.

In [ ]:
from pathlib import Path
import json
import torch

from ml_extension import (
    EXTERNAL_STUDY_ID, FEATURE_SETS, ML_VALUE_COLUMN, TRAIN_STUDY_ID,
    fit_standardizer, load_samples, run_experiments, stratified_development_split,
)

EXPRESSION_CSV = Path("results/expression_values.csv")
OUTPUT_DIR = Path("ml_results")
print("PyTorch", torch.__version__)
print("ML input column:", ML_VALUE_COLUMN)

## Inspect the sample-level data

The tidy expression table generated by `analysis.py` is pivoted to one row per sample. Only samples with a complete feature vector are retained. The model explicitly uses the corrected `log2_expression` column, not the legacy TCGA decimal-stripping transformation stored in `thesis_analysis_value`.

In [ ]:
for experiment, features in FEATURE_SETS.items():
    samples = load_samples(EXPRESSION_CSV, features)
    development = [sample for sample in samples if sample.study_id == TRAIN_STUDY_ID]
    external = [sample for sample in samples if sample.study_id == EXTERNAL_STUDY_ID]
    positive_development = sum(sample.label for sample in development)
    positive_external = sum(sample.label for sample in external)
    print(
        f"{experiment}: {len(development)} TCGA samples "
        f"({positive_development} positive), {len(external)} SU2C samples "
        f"({positive_external} positive)"
    )

## Prevent information leakage

The three-way TCGA split is stratified so both classes occur in training, early-stopping validation and the untouched internal test set. Means and standard deviations are learned **only from the training subset**, then reused unchanged for all other sets. SU2C is not inspected by the optimizer or during model selection.

In [ ]:
primary_samples = load_samples(EXPRESSION_CSV, FEATURE_SETS["primary"])
tcga_samples = [sample for sample in primary_samples if sample.study_id == TRAIN_STUDY_ID]
train_samples, validation_samples, internal_test_samples = stratified_development_split(
    tcga_samples, validation_fraction=0.2, internal_test_fraction=0.2, seed=42
)
standardizer = fit_standardizer(train_samples, FEATURE_SETS["primary"])

print(f"Training samples: {len(train_samples)}")
print(f"Early-stopping validation samples: {len(validation_samples)}")
print(f"Untouched internal-test samples: {len(internal_test_samples)}")
for gene, mean, std in zip(
    standardizer.feature_names, standardizer.means, standardizer.standard_deviations
):
    print(f"{gene:5s}: training mean={mean:.3f}, training SD={std:.3f}")

## Train both models

Each model is a compact multilayer perceptron with one hidden layer, ReLU activation and dropout. Training uses `BCEWithLogitsLoss` with a positive-class weight, AdamW optimization and validation-loss early stopping. A fixed seed makes the run reproducible on the same PyTorch setup.

In [ ]:
metadata = run_experiments(
    EXPRESSION_CSV,
    OUTPUT_DIR,
    seed=42,
    validation_fraction=0.2,
    internal_test_fraction=0.2,
    hidden_units=8,
    dropout=0.1,
    maximum_epochs=500,
    patience=40,
)
print(f"Outputs written to {OUTPUT_DIR.resolve()}")

## Compare internal and external performance

ROC-AUC measures ranking independently of the fixed 0.5 threshold. Balanced accuracy gives equal weight to fusion-positive and fusion-negative samples. Sensitivity is the fraction of fusion-positive samples detected; specificity is the fraction of fusion-negative samples correctly rejected. Precision describes how many positive predictions are correct, F1 combines precision and sensitivity, and a lower Brier score indicates better probability predictions. The internal-test result is not used for early stopping. External performance is the more demanding result because TCGA and SU2C differ biologically and technically.

In [ ]:
from IPython.display import HTML, display

metric_rows = []
for experiment, result in metadata["experiments"].items():
    for split in ("internal_test", "external"):
        metrics = result["metrics"][split]
        metric_rows.append(
            f'''<tr><td>{experiment.replace('_', ' ')}</td>
            <td>{split.replace('_', ' ')}</td><td>{metrics['n']}</td>
            <td>{metrics['positives']}</td><td>{metrics['negatives']}</td>
            <td>{metrics['roc_auc']:.3f}</td>
            <td>{metrics['balanced_accuracy']:.3f}</td>
            <td>{metrics['sensitivity']:.3f}</td>
            <td>{metrics['specificity']:.3f}</td>
            <td>{metrics['precision']:.3f}</td>
            <td>{metrics['f1']:.3f}</td>
            <td>{metrics['brier_score']:.3f}</td></tr>'''
        )

display(HTML(
    '''<div style="overflow-x:auto"><table style="border-collapse:collapse;white-space:nowrap">
    <thead><tr><th>Experiment</th><th>Set</th><th>n</th><th>Positive</th><th>Negative</th>
    <th>ROC-AUC</th><th>Balanced accuracy</th><th>Sensitivity</th><th>Specificity</th>
    <th>Precision</th><th>F1</th><th>Brier score</th></tr></thead><tbody>'''
    + ''.join(metric_rows)
    + '''</tbody></table></div>'''
))

## View the generated figures

The first figure checks optimization and possible overfitting using training and early-stopping validation losses. The second compares untouched internal-test and external ROC curves for the leakage-reduced primary model and the ERG-inclusive positive control.

In [ ]:
from IPython.display import SVG, display

training_figure = (OUTPUT_DIR / "training_curves.svg").read_text(encoding="utf-8")
roc_figure = (OUTPUT_DIR / "roc_curves.svg").read_text(encoding="utf-8")
display(SVG(data=training_figure))
display(SVG(data=roc_figure))

## Interpretation and limitations

This is a small learning experiment, not a clinical model. I would keep the following limitations with any reported result:

1. The primary model has only three expression features and cannot represent the full transcriptome.
2. The fusion labels depend on events returned by cBioPortal; `not detected` is not proof that no relevant alteration exists.
3. TCGA and SU2C 2019 differ in disease stage, sample composition and expression processing, so external performance includes cohort and platform shift.
4. The ERG-inclusive model is a positive control and should not be used to claim general predictive performance.

### Conclusion from the current run

The ERG-excluding primary model showed only weak discrimination on the untouched TCGA test set (ROC-AUC 0.632; balanced accuracy 0.560) and in SU2C (ROC-AUC 0.608). At the fixed 0.5 threshold it classified all 181 SU2C samples as negative, giving sensitivity 0, specificity 1 and balanced accuracy 0.5. The ERG-inclusive positive control retained strong ranking performance (internal ROC-AUC 0.983; SU2C ROC-AUC 0.905), but its external threshold also failed and its Brier score was 0.530.

This combination of preserved ranking and failed absolute probabilities is consistent with substantial domain shift. TCGA and SU2C differ in tumour stage, patient composition, laboratory methods and expression processing. The result does not rule out a biological association, but the classifier is not calibrated or sufficiently transferable for external use. The ERG-inclusive control mainly confirms the expected close relationship between ERG expression and TMPRSS2–ERG status.

A separate project, `rai2_cohort_regression`, addresses a different and more biologically aligned question by predicting continuous RAI2 expression within each cohort. The classifier in this notebook is retained unchanged as the original learning experiment.